# Trace Count v9: Reduced-Capacity v2

v9 keeps the v2 task fixed and changes only model capacity.

- **Data:** prompt length 256; 1–10 needles; 64 noise tokens; 10 marker tokens.
- **Targets:** trace indices and final answers share the same `<1>...<10>` tokens.
- **Models:** one non-thinking model and one thinking model.
- **Small architecture:** 3 layers, 4 heads, `d_model=128`, `MLP=384`.
- **Training:** 10,000 steps, effective batch size 128, no loss weighting.
- **Reliability:** recoverable checkpoints are synced to Google Drive every 2,000 steps.

The goal is to test whether reducing excess capacity creates a regime where explicit
thinking traces improve final-count accuracy or learning speed.

## 1. Environment and Repository Setup

## Google Drive Login

在实验开始时挂载一次 Google Drive。后面的保存 cell 会复用该挂载，因此长实验结束时不会再次弹出登录流程。

In [ ]:
from pathlib import Path
import sys

DRIVE_RESULTS_ROOT = Path(
    "/content/drive/MyDrive/Colab_Notebooks/CoT_Counting/"
    "Synthetic_CoT_NiaH_Count/colab_results"
)
DRIVE_MOUNTED = False

def ensure_google_drive_mounted() -> bool:
    global DRIVE_MOUNTED
    if not ("google.colab" in sys.modules or Path("/content").exists()):
        print("Not in Colab; Google Drive mount skipped.")
        return False
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    DRIVE_RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
    DRIVE_MOUNTED = True
    print("Google Drive ready:", DRIVE_RESULTS_ROOT)
    return True

ensure_google_drive_mounted()

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/Twist-Shan/Synthetic_CoT_NiaH_Count.git"
IN_COLAB = "google.colab" in sys.modules or Path("/content").exists()
PULL_REPO = True
INSTALL_DEPS = False

if IN_COLAB:
    repo_dir = Path("/content/Synthetic_CoT_NiaH_Count")
    cwd = Path.cwd()
    if (cwd / ".git").exists():
        repo_dir = cwd
    elif not repo_dir.exists():
        subprocess.run(["git", "clone", REPO_URL, str(repo_dir)], check=True)
    os.chdir(repo_dir)
    if PULL_REPO and (repo_dir / ".git").exists():
        subprocess.run(["git", "pull"], check=False)

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

if INSTALL_DEPS:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "transformers>=4.40", "pandas", "matplotlib", "tqdm"],
        check=True,
    )

import pandas as pd
import torch
from IPython.display import Image, Markdown, display

display(Markdown(f"**Repo root:** `{ROOT}`  \n**Device:** `{torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}`"))

## 2. Runtime Settings

In [ ]:
PRESET = "main"  # use "debug" for a four-step pipeline check
OUT_ROOT = "runs/trace_count_v9_small_capacity"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SKIP_COMPLETED = True

LIVE_CHECKPOINT_ROOT = DRIVE_RESULTS_ROOT / "v9_live_checkpoints" if DRIVE_MOUNTED else None
if LIVE_CHECKPOINT_ROOT is not None:
    LIVE_CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)

print({
    "PRESET": PRESET,
    "OUT_ROOT": OUT_ROOT,
    "DEVICE": DEVICE,
    "LIVE_CHECKPOINT_ROOT": str(LIVE_CHECKPOINT_ROOT),
})

## 3. Capacity Check

The table below constructs both architectures on CPU and reports their exact trainable
parameter counts. The v2 reference uses 4 layers, 4 heads, `d_model=256`, and `MLP=1024`;
the v9 row changes only capacity while retaining the same vocabulary and context length.

In [ ]:
from dataclasses import replace

from synthetic_counting_extensions.v7_v8_sweeps import (
    Vocab,
    build_model_cfg,
    preset_configs,
)
from synthetic_niah_v5.model import make_model

settings = preset_configs("v9", PRESET)
v9_cfg = settings[0]

def exact_parameter_count(cfg) -> int:
    cpu_cfg = replace(cfg, device="cpu")
    vocab = Vocab(cpu_cfg)
    model = make_model(build_model_cfg(cpu_cfg, vocab), "cpu")
    count = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    del model
    return int(count)

v2_reference = replace(
    v9_cfg,
    n_layer=4,
    n_head=4,
    n_embd=256,
    n_inner=1024,
    checkpoint_every=0,
)
v2_params = exact_parameter_count(v2_reference)
v9_params = exact_parameter_count(v9_cfg)

capacity_table = pd.DataFrame([
    {
        "model": "v2 reference",
        "layers": 4,
        "heads": 4,
        "d_model": 256,
        "head_dim": 64,
        "mlp_width": 1024,
        "trainable_parameters": v2_params,
        "fraction_of_v2": 1.0,
    },
    {
        "model": "v9 reduced",
        "layers": v9_cfg.n_layer,
        "heads": v9_cfg.n_head,
        "d_model": v9_cfg.n_embd,
        "head_dim": v9_cfg.n_embd // v9_cfg.n_head,
        "mlp_width": v9_cfg.n_inner,
        "trainable_parameters": v9_params,
        "fraction_of_v2": v9_params / v2_params,
    },
])
display(capacity_table)
display(pd.DataFrame([vars(cfg) | {"effective_batch_size": cfg.effective_batch_size} for cfg in settings]))

## 4. Train and Evaluate

This runs exactly two independently initialized models. At steps 2k, 4k, 6k, 8k,
and 10k, each model writes a recoverable checkpoint containing model weights,
optimizer state, step, and RNG state, then immediately syncs it to Drive.

In [ ]:
from synthetic_counting_extensions.v7_v8_sweeps import run_sweep

display(Markdown("**Training runs:** `1 setting × 2 models = 2`"))
combined = run_sweep(
    "v9",
    PRESET,
    OUT_ROOT,
    skip_completed=SKIP_COMPLETED,
    device=DEVICE,
    checkpoint_sync_root=LIVE_CHECKPOINT_ROOT,
)
display(combined)

## 5. Results

In [ ]:
result_root = Path(OUT_ROOT)
display(combined.groupby(["mode"], as_index=False)[["accuracy", "mae"]].mean())

for run in sorted(result_root.glob("v9_*")):
    display(Markdown(f"### `{run.name}`"))
    for figure_name in ["accuracy_by_count.png", "accuracy_heatmap.png", "accuracy_by_validation_split.png"]:
        figure = run / "figures" / figure_name
        if figure.exists():
            display(Image(filename=str(figure)))
    report = run / "report" / "report.html"
    if report.exists():
        display(Markdown(f"Generated report: `{report}`"))

## 6. Save Complete Results to Google Drive

In [ ]:
from datetime import datetime

SAVE_TO_DRIVE = True
DRIVE_SAVE_COMPLETED = False

if SAVE_TO_DRIVE and ensure_google_drive_mounted():
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    source = Path(OUT_ROOT)
    destination = DRIVE_RESULTS_ROOT / f"v9_small_capacity_{PRESET}_{timestamp}"
    shutil.copytree(source, destination, dirs_exist_ok=True)
    manifest = {
        "experiment": "v9",
        "preset": PRESET,
        "source": str(source),
        "live_checkpoint_root": str(LIVE_CHECKPOINT_ROOT),
        "saved_at": timestamp,
    }
    (destination / "drive_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    DRIVE_SAVE_COMPLETED = True
    display(Markdown(f"**Saved result bundle:** `{destination}`"))
else:
    print("Drive result save skipped.")

## 7. Optional GitHub Result Upload

In [ ]:
PUSH_RESULTS_TO_GITHUB = False
GIT_BRANCH = "v9-results"

if PUSH_RESULTS_TO_GITHUB:
    subprocess.run(["git", "checkout", "-B", GIT_BRANCH], check=True)
    subprocess.run(["git", "add", OUT_ROOT], check=True)
    subprocess.run(["git", "commit", "-m", "Add v9 reduced-capacity results"], check=False)
    subprocess.run(["git", "push", "-u", "origin", GIT_BRANCH], check=True)
else:
    print("PUSH_RESULTS_TO_GITHUB is False")

## 8. Optional Runtime Disconnect

In [ ]:
AUTO_DISCONNECT_AFTER_DRIVE_SAVE = False

if AUTO_DISCONNECT_AFTER_DRIVE_SAVE and DRIVE_SAVE_COMPLETED and IN_COLAB:
    from google.colab import runtime
    print("Drive save confirmed; disconnecting Colab runtime.")
    runtime.unassign()
else:
    print("Runtime remains connected.")